# 09 Bundles and Cross-Trace Comparison

This notebook audits `tl.bundle`, which groups several `Trace` objects so you can compare aligned sites across runs. The Super* family is the set of cross-trace views, such as `SuperOp`, that represent one site across bundle members.

We run the same model on three inputs, build a bundle, and compare activations at matching layers.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

In [ ]:
import torch
from torch import nn
import torchlens as tl

torch.manual_seed(41)


class BundleNet(nn.Module):
    """Tiny shared model for cross-trace alignment."""

    def __init__(self) -> None:
        """Initialize layers."""

        super().__init__()
        self.stem = nn.Linear(3, 4)
        self.head = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the model.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        torch.Tensor
            Two-column output.
        """

        hidden = torch.relu(self.stem(x))
        return self.head(hidden)


model = BundleNet().eval()
inputs = {
    "low": torch.tensor([[0.1, 0.2, 0.3]]),
    "mid": torch.tensor([[0.5, 0.2, -0.1]]),
    "high": torch.tensor([[1.0, -0.5, 0.25]]),
}
traces = {name: tl.trace(model, value, intervention_ready=True) for name, value in inputs.items()}
for name, trace in traces.items():
    print(f"{name:4s} output={trace[trace.output_layers[0]].out.flatten().tolist()}")

`tl.bundle` accepts a mapping of names to traces. The bundle preserves member names and can apply a function to each trace.

In [ ]:
bundle = tl.bundle(traces, baseline="low")
print(f"bundle type: {type(bundle).__name__}")
print(f"names: {bundle.names}")
print(f"baseline: {bundle.baseline_name}")
print(f"layer counts: {bundle.apply(lambda trace: len(trace.layer_list))}")
print(f"structurally consistent: {bundle.is_structurally_consistent}")

Shared labels are the alignment keys. Because all members come from the same model graph, the main layer labels are shared.

In [ ]:
print(f"shared layers: {bundle.shared_layer_labels}")
print(f"divergent layers: {bundle.divergent_layer_labels}")
print(f"shared ops: {bundle.shared_op_labels[:4]}")
print(f"relationship low/mid: {bundle.relationship('low', 'mid')}")

A `SuperOp` or `SuperLayer` is a cross-trace view of one aligned site. It exposes represented members by bundle name.

In [ ]:
relu_label = next(label for label in bundle.shared_layer_labels if label.startswith("relu"))
super_relu = bundle.layers[relu_label]
print(f"super view type: {type(super_relu).__name__}")
print(f"representative label: {super_relu.label}")
print(f"member names: {super_relu.members.keys()}")
for name, layer in super_relu.members.items():
    print(f"{name:4s} relu sum={layer.out.sum().item():.6f} shape={tuple(layer.out.shape)}")

`compare_at(site)` returns a pairwise distance matrix for one resolved site. Here, the ReLU activation differs across inputs.

In [ ]:
matrix = bundle.compare_at(relu_label)
print(f"compare_at({relu_label!r}) shape: {tuple(matrix.shape)}")
print(matrix)

`most_changed` ranks sites by change from the baseline member. This is a compact way to find where traces diverge most.

In [ ]:
for label, score in bundle.most_changed(top_k=4):
    print(f"{label:16s} score={score:.6f}")

Bundles can also be saved and loaded through the unified `.tlspec` loader.

In [ ]:
import shutil

ARTIFACT_DIR = REPO_ROOT / "notebooks" / "audit" / "_artifacts"
bundle_path = ARTIFACT_DIR / "09_bundle_roundtrip.tlspec"
if bundle_path.exists():
    shutil.rmtree(bundle_path)

bundle.save(bundle_path, level="portable", overwrite=True)
loaded_bundle = tl.load(bundle_path)
print(f"loaded type: {type(loaded_bundle).__name__}")
print(f"loaded names: {loaded_bundle.names}")
print(f"loaded baseline: {loaded_bundle.baseline_name}")
print(f"loaded compare shape: {tuple(loaded_bundle.compare_at(relu_label).shape)}")

> NOTE: The Super* family is exposed through bundle accessors such as `bundle.layers[...]`, `bundle.ops[...]`, `bundle.modules[...]`, and `bundle.grad_fns[...]`. This build does not expose separate top-level `SuperOp` constructor helpers.

## Bundle Lifecycle, Interventions, and Super Views
Bundles are mutable containers: `add`, `remove`, `clear`, and `fork` manage members. Bundle-level `do`/`replay` can apply the same intervention across traces, `diff_pair` compares two members, and non-layer Super views expose aligned modules, params, and grad functions.

In [ ]:
lifecycle = bundle.fork(name="lifecycle_demo")
extra = tl.trace(model, torch.tensor([[0.0, 0.0, 0.0]]), intervention_ready=True)
lifecycle.add(extra, names="zero")
print(f"after add: {lifecycle.names}")
removed = lifecycle.remove("zero")
print(f"removed type: {type(removed).__name__}; names now: {lifecycle.names}")
kept = lifecycle.fork(name="kept_only")
kept.remove_except(["low", "mid"])
print(f"remove_except names: {kept.names}")
cleared = lifecycle.fork(name="cleared")
cleared.clear()
print(f"after clear count: {len(cleared.names)}")

intervened_bundle = bundle.fork(name="bundle_zero_relu")
try:
    intervened_bundle.do(tl.func("relu"), tl.zero_ablate(), engine="replay")
    before = bundle.layers[relu_label].members["high"].out.sum().item()
    after = intervened_bundle.layers[relu_label].members["high"].out.sum().item()
    print(f"bundle.do relu high sum: {before:.6f} -> {after:.6f}")
except Exception as exc:
    print(f"> NOTE: bundle.do/replay gap: {type(exc).__name__}: {exc}")

try:
    pair = bundle.diff_pair("low", "high")
    print(f"diff_pair type: {type(pair).__name__}")
except Exception as exc:
    print(f"> NOTE: diff_pair gap: {type(exc).__name__}: {exc}")

module_key = next(iter(bundle.modules))
param_key = next(iter(bundle.params))
print(f"SuperModule {module_key}: {type(bundle.modules[module_key]).__name__}")
print(f"SuperParam {param_key}: {type(bundle.params[param_key]).__name__}")

backward_traces = {}
for name, value in inputs.items():
    value = value.detach().clone().requires_grad_(True)
    backward_trace = tl.trace(model, value, backward_ready=True, gradients_to_save="all")
    backward_trace.log_backward(backward_trace[backward_trace.output_layers[0]].out.sum())
    backward_traces[name] = backward_trace
backward_bundle = tl.bundle(backward_traces, baseline="low")
if len(backward_bundle.grad_fns):
    grad_key = next(iter(backward_bundle.grad_fns))
    print(f"SuperGradFn {grad_key}: {type(backward_bundle.grad_fns[grad_key]).__name__}")
else:
    print("> NOTE: no SuperGradFn entries were available after backward capture.")
print(f"bundle help excerpt: {bundle.help().splitlines()[0]}")

## 🔧 Sandbox

Mini-experiment: change `sandbox_site`, `sandbox_member`, and `sandbox_compare`. Expected observation: the selected member shows one tensor, while bundle comparison summarizes deltas across all members.

In [ ]:
sandbox_site = relu_label
sandbox_member = "high"
sandbox_compare = True
sandbox_super = bundle.layers[sandbox_site]
member_layer = sandbox_super.members[sandbox_member]
print(f"site: {sandbox_site}")
print(f"member {sandbox_member} values={member_layer.out.detach().flatten()[:3].tolist()}")
print(f"member count: {len(sandbox_super.members)}")
if sandbox_compare:
    print(bundle.compare_at(sandbox_site))